***

*Course:* [Math 535](https://people.math.wisc.edu/~roch/mmids/) - Mathematical Methods in Data Science (MMiDS)  
*Chapter:* 6-Optimization theory and algorithms   
*Author:* [Sebastien Roch](https://people.math.wisc.edu/~roch/), Department of Mathematics, University of Wisconsin-Madison  
*Updated:* Feb 14, 2024   
*Copyright:* &copy; 2024 Sebastien Roch

***

In [ ]:
# You will need the files:
#     * mmids.py
#     * advertising.csv 
# from https://github.com/MMiDS-textbook/MMiDS-textbook.github.io/tree/main/utils
#
# IF RUNNING ON GOOGLE COLAB (RECOMMENDED):
# "Upload to session storage" from the Files tab on the left
# Alternative instructions: https://colab.research.google.com/notebooks/io.ipynb

In [ ]:
# FOR e-BOOK ONLY
import os, sys
sys.path.insert(0, os.path.abspath('../../utils')) # use directory to mmids.py

In [ ]:
# PYTHON 3
import numpy as np
from numpy import linalg as LA
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import mmids
seed = 535
rng = np.random.default_rng(seed)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# FOR TeX ONLY
#plt.rcParams['figure.figsize'] = [4.00, 2.25] # for high-def figs
#plt.rcParams['figure.dpi'] = 200 # for high-def figs

## Motivating example:  deciphering handwriting

**Figure:** Helpful map of ML by scitkit-learn ([Source](https://scikit-learn.org/stable/tutorial/machine_learning_map/index.html))

![ml-cheat-sheet](https://scikit-learn.org/stable/_static/ml_map.png)

$\bowtie$

We now turn to classification.

Quoting [Wikipedia](https://en.wikipedia.org/wiki/Statistical_classification):

> In machine learning and statistics, classification is the problem of identifying to which of a set of categories (sub-populations) a new observation belongs, on the basis of a training set of data containing observations (or instances) whose category membership is known. Examples are assigning a given email to the "spam" or "non-spam" class, and assigning a diagnosis to a given patient based on observed characteristics of the patient (sex, blood pressure, presence or absence of certain symptoms, etc.). Classification is an example of pattern recognition. In the terminology of machine learning, classification is considered an instance of supervised learning, i.e., learning where a training set of correctly identified observations is available.

We will illustrate this problem on the [MNIST](https://en.wikipedia.org/wiki/MNIST_database) dataset. Quoting [Wikipedia](https://en.wikipedia.org/wiki/MNIST_database) again:

> The MNIST database (Modified National Institute of Standards and Technology database) is a large database of handwritten digits that is commonly used for training various image processing systems. The database is also widely used for training and testing in the field of machine learning. It was created by "re-mixing" the samples from NIST's original datasets. The creators felt that since NIST's training dataset was taken from American Census Bureau employees, while the testing dataset was taken from American high school students, it was not well-suited for machine learning experiments. Furthermore, the black and white images from NIST were normalized to fit into a 28x28 pixel bounding box and anti-aliased, which introduced grayscale levels. The MNIST database contains 60,000 training images and 10,000 testing images. Half of the training set and half of the test set were taken from NIST's training dataset, while the other half of the training set and the other half of the test set were taken from NIST's testing dataset.

**Figure:** MNIST sample images ([Source](https://commons.wikimedia.org/wiki/File:MnistExamples.png))

![MNIST sample images](https://upload.wikimedia.org/wikipedia/commons/2/27/MnistExamples.png)

$\bowtie$

<!--TEX

![MNIST sample images ([Source](https://commons.wikimedia.org/wiki/File:MnistExamples.png))](https://upload.wikimedia.org/wikipedia/commons/2/27/MnistExamples.png)

-->

We first load the data and convert it to an appropriate matrix representation. The data can be accessed with [`torchvision.datasets.MNIST`](https://pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html).

In [ ]:
import torch
from torchvision import datasets, transforms

In [ ]:
# Download and load the MNIST dataset
mnist = datasets.MNIST(root='./data', 
                       train=True, 
                       download=True, 
                       transform=transforms.ToTensor())

# Convert the dataset to a PyTorch DataLoader
train_loader = torch.utils.data.DataLoader(mnist, 
                                           batch_size=len(mnist), 
                                           shuffle=False)

# Extract images and labels from the DataLoader
imgs, labels = next(iter(train_loader))
imgs = imgs.squeeze().numpy()
labels = labels.numpy()

The [`squeeze()`](https://pytorch.org/docs/stable/generated/torch.Tensor.squeeze.html) above removes the color dimension in the image, which is grayscale. The [`numpy()`](https://pytorch.org/docs/stable/generated/torch.Tensor.numpy.html) converts the PyTorch tensors into Numpy arrays. See [`torch.utils.data.DataLoader`](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) for details on the data loading. We will say more about PyTorch later in this chapter.

For example, the first image and its label are:

In [ ]:
plt.figure()
plt.imshow(imgs[0])
plt.show()

In [ ]:
labels[0]

For now, we look at a subset of the samples: the 0's and 1's.

In [ ]:
# Filter out images with labels 0 and 1
mask = (labels == 0) | (labels == 1)
imgs01 = imgs[mask]
labels01 = labels[mask]

In this new dataset, the first sample is:

In [ ]:
plt.figure()
plt.imshow(imgs01[0])
plt.show()

In [ ]:
labels01[0]

Next, we transform the images into vectors. For this we use the [`reshape()`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.reshape.html) function, which changes the dimensions of an array without changing its data. Here the first dimension, which runs across the samples, remains of the same length `len(imgs01)`. The `-1` is understood to be whatever is needed to "fit" the remaining dimensions, here $28 \times 28 = 784$. In other words, we are effectively "flattening" each $28 \times 28$ image into a single vector of length $784$. 

In [ ]:
imgs01.shape

In [ ]:
X = imgs01.reshape(len(imgs01), -1)

In [ ]:
X.shape

In [ ]:
y = labels01

The input data is now of the form $\{(\mathbf{x}_i, y_i) : i=1,\ldots, n\}$ where $\mathbf{x}_i \in \mathbb{R}^d$ are the features and $y_i \in \{0,1\}$ is the label. Above we use the matrix representation $X \in \mathbb{R}^{d \times n}$ with columns $\mathbf{x}_i$, $i = 1,\ldots, n$ and $\mathbf{y} = (y_1, \ldots, y_n)^T \in \{0,1\}^n$. 

Our goal: 

> learn a classifier from the examples $\{(\mathbf{x}_i, y_i) : i=1,\ldots, n\}$, that is, a function $\hat{f} : \mathbb{R}^d \to \mathbb{R}$ such that $\hat{f}(\mathbf{x}_i) \approx y_i$.

We may want to enforce that the output is in $\{0,1\}$ as well. This problem is referred to as [binary classification](https://en.wikipedia.org/wiki/Binary_classification).

A natural approach to this type of [supervised learning](https://en.wikipedia.org/wiki/Supervised_learning) problem is to define two objects:

1. **Family of classifiers:** A class $\widehat{\mathcal{F}}$ of classifiers from which to pick $\hat{f}$.

2. **Loss function:** A loss function $\ell(\hat{f}, (\mathbf{x},y))$ which quantifies how good of a fit $\hat{f}(\mathbf{x})$ is to $y$.

Our goal is then to solve

$$
\min_{\hat{f} \in \widehat{\mathcal{F}}} \frac{1}{n} \sum_{i=1}^n \ell(\hat{f}, (\mathbf{x}_i, y_i)),
$$

that is, we seek to find a classifier among $\widehat{\mathcal{F}}$ that minimizes the average loss over the examples.

For instance, in [logistic regression](https://en.wikipedia.org/wiki/Logistic_regression), we consider linear classifiers of the form

$$
\hat{f}(\mathbf{x})
= \sigma(\mathbf{x}^T \boldsymbol{\theta})
\qquad
\text{with}
\qquad
\sigma(t) = \frac{1}{1 + e^{-t}}
$$

where $\boldsymbol{\theta} \in \mathbb{R}^d$ is a parameter vector. And we use the [cross-entropy loss](https://en.wikipedia.org/wiki/Cross_entropy#Cross-entropy_loss_function_and_logistic_regression)

$$
\ell(\hat{f}, (\mathbf{x}, y))
= -  y \log(\sigma(\mathbf{x}^T \boldsymbol{\theta}))
- (1-y) \log(1- \sigma(\mathbf{x}^T \boldsymbol{\theta})).
$$

In parametric form, the problem boils down to

$$
\min_{\boldsymbol{\theta} \in \mathbb{R}^d}
- \frac{1}{n} \sum_{i=1}^n y_i \log(\sigma(\mathbf{x}_i^T \boldsymbol{\theta}))
- \frac{1}{n} \sum_{i=1}^n (1-y_i) \log(1- \sigma(\mathbf{x}_i^T \boldsymbol{\theta})).
$$

To obtain a prediction in $\{0,1\}$ here, we could cutoff $\hat{f}(\mathbf{x})$ at a threshold $\tau \in [0,1]$, that is, return $\mathbf{1}\{\hat{f}(\mathbf{x}) > \tau\}$.

We will explain in a later chapter where this choice comes from.

The purpose of this chapter is to develop some of the mathematical theory and algorithms needed to solve this type of optimization formulation.